# Compliant mechanism topology optimization

Reproduction of the 104-line MATLAB force-inverter example in Section 5.1.5
of Bendsøe and Sigmund (2004). The reference call is
`topm(40,20,0.3,3.0,1.2)`, with reference objective `-1.1131886`.

> Bendsøe, M. P., and Sigmund, O. *Topology Optimization: Theory, Methods,
> and Applications*. Springer, 2004.

In [ ]:
import jax
import jax.numpy as np

jax.config.update("jax_enable_x64", True)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax_fem import logger
from PIL import Image as PILImage

logger.setLevel("WARNING")

from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper

from topax.filter import build_conv_filter
from topax.problem import TopOptProblem

In [ ]:
class CompliantMechanism(TopOptProblem):
    """Linear-elastic half force inverter with input and output springs."""

    def custom_init(self):
        self.fe = self.fes[0]
        self.fe.flex_inds = np.arange(len(self.fe.cells))

        points = self.fe.points
        Lx = np.max(points[:, 0])
        Ly = np.max(points[:, 1])

        def input_location(point):
            return np.logical_and(
                np.isclose(point[0], 0., atol=1e-5),
                np.isclose(point[1], Ly, atol=1e-5),
            )

        def output_location(point):
            return np.logical_and(
                np.isclose(point[0], Lx, atol=1e-5),
                np.isclose(point[1], Ly, atol=1e-5),
            )

        self.input_node = self.add_point_load(
            input_location, np.array([1., 0.])
        )
        output_mask = jax.vmap(output_location)(points)
        output_nodes = onp.asarray(np.argwhere(output_mask).reshape(-1))
        if len(output_nodes) != 1:
            raise ValueError(
                "The output location must select exactly one mesh node."
            )
        self.output_node = int(output_nodes[0])

        self.spring_nodes = onp.array(
            [self.input_node, self.output_node], dtype=onp.int32
        )
        self.spring_stiffness = onp.array([0.1, 0.1])
        spring_dofs = self.spring_nodes * self.fe.vec + self.offset[0]
        self.I = onp.hstack((self.I, spring_dofs))
        self.J = onp.hstack((self.J, spring_dofs))

    def get_tensor_map(self):
        def stress(u_grad, xPhys):
            E = xPhys**3
            nu = 0.3
            mu = E / (2. * (1. + nu))
            lmbda = E * nu / ((1. + nu) * (1. - 2. * nu))
            lmbda = 2. * mu * lmbda / (lmbda + 2. * mu)
            epsilon = 0.5 * (u_grad + u_grad.T)
            sigma = lmbda * np.trace(epsilon) * np.eye(self.dim) + 2. * mu * epsilon
            return sigma
        return stress

    def set_params(self, params):
        full_params = np.ones((self.fe.num_cells, params.shape[1]))
        full_params = full_params.at[self.fe.flex_inds].set(params)
        thetas = np.repeat(full_params[:, None, :], self.fe.num_quads, axis=1)
        self.full_params = full_params
        self.internal_vars = [thetas]

    def _add_spring_residual(self, res_list, sol_list):
        res_list = list(res_list)
        spring_displacements = sol_list[0][self.spring_nodes, 0]
        spring_forces = self.spring_stiffness * spring_displacements
        res_list[0] = res_list[0].at[self.spring_nodes, 0].add(spring_forces)
        return res_list

    def compute_residual(self, sol_list):
        res_list = super().compute_residual(sol_list)
        return self._add_spring_residual(res_list, sol_list)

    def newton_update(self, sol_list):
        res_list = super().newton_update(sol_list)
        self.V = onp.hstack((self.V, self.spring_stiffness))
        return self._add_spring_residual(res_list, sol_list)


def prep_fem(Nx, Ny, Lx, Ly):
    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(Nx, Ny, domain_x=Lx, domain_y=Ly)
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type], ele_type)

    def symmetry(point):
        return np.isclose(point[1], Ly, atol=1e-5)

    def fixed_left_bottom(point):
        return np.logical_and(
            np.isclose(point[0], 0., atol=1e-5),
            point[1] <= Ly / Ny + 1e-5,
        )

    zero = lambda point: 0.
    dirichlet_bc_info = [
        [symmetry, fixed_left_bottom, fixed_left_bottom],
        [1, 0, 1],
        [zero, zero, zero],
    ]

    problem = CompliantMechanism(
        mesh,
        vec=2,
        dim=2,
        ele_type=ele_type,
        dirichlet_bc_info=dirichlet_bc_info,
    )

    solver_options = {'petsc_solver': {'ksp_type': 'preonly', 'pc_type': 'lu'}}
    fwd_pred = ad_wrapper(
        problem,
        solver_options=solver_options,
        adjoint_solver_options=solver_options,
    )
    return fwd_pred, problem

In [ ]:
# SETUP
vf = 0.3
rmin = 1.2

Nx, Ny = 40, 20
Lx, Ly = Nx, Ny
fwd_pred, problem = prep_fem(Nx, Ny, Lx, Ly)


def J_total(xPhys):
    u = fwd_pred(xPhys)[0]
    return u[problem.output_node, 0]


def volume_constraint(xPhys):
    return np.sum(xPhys) - vf * xPhys.size


H, Hs = build_conv_filter(problem, rmin=rmin)

def sensitivity_filter(dc, x):
    dc_col = dc.reshape(-1, 1)
    x_col = x.reshape(-1, 1)
    numerator = H @ (dc_col * x_col)
    denominator = Hs * np.maximum(x_col, 1e-3)
    return (numerator / denominator).reshape(dc.shape)


def oc_update(x, dc):
    l1 = 0.0
    l2 = 1e5
    move = 0.1
    while (l2 - l1) / (l2 + l1) > 1e-4 and l2 > 1e-40:
        lmid = 0.5 * (l2 + l1)
        ratio = np.maximum(1e-10, -dc / lmid)
        xnew = np.maximum(
            0.001,
            np.maximum(
                x - move,
                np.minimum(1., np.minimum(x + move, x * ratio**0.3)),
            ),
        )
        if np.sum(xnew) > vf * Nx * Ny:
            l1 = lmid
        else:
            l2 = lmid
    return xnew


x0 = vf * np.ones((Nx * Ny, 1))

In [ ]:
# OPTIMIZATION LOOP
loop = 0
change = 1
xnew = x0
frames = []
while change > 0.01:
    loop += 1
    J, dJ = jax.value_and_grad(J_total)(xnew)
    xold = xnew.copy()
    dJ = sensitivity_filter(dJ, xold)
    xnew = oc_update(xold, dJ)
    xPhys = xnew
    vol = np.mean(xPhys)
    change = np.max(np.abs(xnew - xold))
    print(f' It.:{loop:5d}, Obj.:{J:11.4f}, Vol.:{vol:7.3f}, ch.:{change:7.3f}')
    field = onp.flip(xPhys.reshape(Ny, Nx, order='F'), axis=0)
    frames.append(onp.asarray(field))

In [ ]:
# SAVE OPTIMIZATION HISTORY
output_path = Path("docs/imgs/example_topopt_mems.gif")
output_path.parent.mkdir(parents=True, exist_ok=True)

gif_frames = []
for field in frames:
    rgba = plt.get_cmap("gray_r")(onp.clip(field, 0.0, 1.0), bytes=True)
    image = PILImage.fromarray(rgba)
    image = image.resize((Nx * 8, Ny * 8), PILImage.Resampling.NEAREST)
    gif_frames.append(image)

gif_frames[0].save(
    output_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
)
display(DisplayImage(filename=str(output_path)))